# Automated Deepfake Detection Training on Google Colab

This notebook automates the process of downloading the necessary datasets and training the deepfake detection model. Follow the steps below:

## Step 1: Mount Google Drive

Run the following cell to mount your Google Drive. You will be prompted to authorize Colab to access your Drive. The trained model will be saved to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Dependencies

This cell installs the required Python libraries, including `gdown` for downloading the datasets.

In [ ]:
!pip install -q torch torchvision torchaudio albumentations scikit-learn matplotlib seaborn pandas opencv-python-headless huggingface_hub


## Step 2.5: Setup (Optional)

Hugging Face datasets used below are public and do not require login. You can skip any setup and just run the code.

In [ ]:
# Kaggle setup removed for simplicity. Hugging Face is used instead.
print('Ready to download from Hugging Face.')

## Step 3: Download Datasets (Zero-Setup)

The code below will automatically download a high-quality Deepfake Detection dataset from Hugging Face. **No rules to accept, no Kaggle account needed.**

In [ ]:
import os
from huggingface_hub import snapshot_download
import zipfile
import shutil

DATA_DIR = 'data/Combined_Dataset'
os.makedirs(DATA_DIR, exist_ok=True)

print("🚀 Starting Robust Download...")
snapshot_download(repo_id='yashduhan/DeepFakeDetection', repo_type='dataset', local_dir=DATA_DIR)

# Force extraction of any zip files found
for item in os.listdir(DATA_DIR):
    if item.endswith('.zip'):
        zip_path = os.path.join(DATA_DIR, item)
        print(f"📦 Extracting {item} to {DATA_DIR}...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(DATA_DIR)
            print(f"✅ Successfully extracted {item}")
        except Exception as e:
            print(f"❌ Failed to extract {item}: {e}")

print("\n🔍 Verifying folders...")
found_data = False
for root, dirs, files in os.walk(DATA_DIR):
    if 'REAL' in dirs and 'FAKE' in dirs:
        print(f"✅ SUCCESS: Found data folders at {root}")
        found_data = True
        break

if not found_data:
    print("❌ FAILED: Could not find REAL/FAKE folders. Listing current files:")
    !ls -R {DATA_DIR} | head -n 20


## Step 4: Train the Model

This cell contains the complete training script. It will train the model using the downloaded datasets and save the `best_model.pth` file to your Google Drive.

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision import models
from tqdm import tqdm
import time
import json
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
import matplotlib.pyplot as plt

# --- Data Pipeline: FaceExtractor ---
class FaceExtractor:
    def __init__(self, cascade_path=cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'):
        self.face_cascade = cv2.CascadeClassifier(cascade_path)
        self.target_size = (224, 224)
        
    def extract_face(self, image_bgr, margin=20):
        gray_image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
        faces = self.face_cascade.detectMultiScale(gray_image, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
        if len(faces) == 0:
            h, w = image_bgr.shape[:2]
            cx, cy = w // 2, h // 2
            size = min(w, h)
            cropped = image_bgr[cy - size//2 : cy + size//2, cx - size//2 : cx + size//2]
            return cv2.resize(cropped, self.target_size)
        faces = sorted(faces, key=lambda x: x[2] * x[3], reverse=True)
        x, y, w, h = faces[0]
        h_img, w_img = image_bgr.shape[:2]
        x1 = max(0, x - margin)
        y1 = max(0, y - margin)
        x2 = min(w_img, x + w + margin)
        y2 = min(h_img, y + h + margin)
        face_crop = image_bgr[y1:y2, x1:x2]
        return cv2.resize(face_crop, self.target_size)

# --- Data Pipeline: Augmentations ---
def get_train_transforms(image_size=224):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.HorizontalFlip(p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 7)),
            A.MotionBlur(blur_limit=7)
        ], p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0),
        ToTensorV2()
    ])

def get_val_transforms(image_size=224):
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0),
        ToTensorV2()
    ])

# --- Data Pipeline: Dataset Loader ---
class DeepfakeDataset(Dataset):
    def __init__(self, data_root, transform=None, extract_face=False):
        self.image_paths = []
        self.labels = []
        self.transform = transform
        self.extract_face = extract_face
        if self.extract_face:
            self.face_extractor = FaceExtractor()
        
        print(f"Searching for data in {data_root}...")
        found = False
        for root, dirs, files in os.walk(data_root):
            if 'REAL' in [d.upper() for d in dirs] and 'FAKE' in [d.upper() for d in dirs]:
                print(f"✅ Found REAL/FAKE folders at: {root}")
                found = True
                real_folder = [d for d in dirs if d.upper() == 'REAL'][0]
                fake_folder = [d for d in dirs if d.upper() == 'FAKE'][0]
                
                for label, folder in enumerate([real_folder, fake_folder]):
                    path = os.path.join(root, folder)
                    for img in os.listdir(path):
                        if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                            self.image_paths.append(os.path.join(path, img))
                            self.labels.append(label)
                break
        
        if not found:
            print(f"❌ ERROR: Could not find any REAL/FAKE folders in {data_root}")
        else:
            print(f"📊 Loaded {len(self.image_paths)} images.")
        
    def __len__(self):
        return len(self.image_paths)
        
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = cv2.imread(img_path)
        if self.extract_face:
            image = self.face_extractor.extract_face(image)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        return image, label

# --- Model: TransferDeepfakeModel ---
class TransferDeepfakeModel(nn.Module):
    def __init__(self, target_model='efficientnet_b0', freeze_base=True, pretrained=True):
        super(TransferDeepfakeModel, self).__init__()
        self.target_model = target_model
        if target_model == 'efficientnet_b0':
            if pretrained:
                weights = models.EfficientNet_B0_Weights.DEFAULT
            else:
                weights = None
            self.base_model = models.efficientnet_b0(weights=weights)
            if freeze_base:
                for param in self.base_model.parameters():
                    param.requires_grad = False
            num_ftrs = self.base_model.classifier[1].in_features
            self.base_model.classifier = nn.Sequential(
                nn.Dropout(p=0.5, inplace=True),
                nn.Linear(num_ftrs, 1)
            )
        else:
            raise ValueError(f"Model {target_model} is not currently supported.")

    def forward(self, x):
        return self.base_model(x)

    def predict(self, x):
        logits = self.forward(x)
        return torch.sigmoid(logits)
        
    def unfreeze_base_model(self, num_layers=20):
        for param in self.base_model.parameters():
            param.requires_grad = False
        for param in self.base_model.classifier.parameters():
            param.requires_grad = True
        if self.target_model == 'efficientnet_b0':
            total_blocks = len(self.base_model.features)
            for i in range(total_blocks - num_layers, total_blocks):
                for param in self.base_model.features[i].parameters():
                    param.requires_grad = True

# --- Training: train_model ---
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

def train_model(model, train_loader, val_loader, epochs, lr, device, save_dir):
    model = model.to(device)
    os.makedirs(save_dir, exist_ok=True)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            train_bar.set_postfix({'loss': loss.item()})
            
        epoch_loss = running_loss / len(train_loader.dataset)
        
        model.eval()
        val_loss = 0.0
        all_labels = []
        all_preds = []
        all_probs = []
        
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")
            for inputs, labels in val_bar:
                inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                probs = torch.sigmoid(outputs)
                preds = (probs > 0.5).float()
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                val_bar.set_postfix({'val_loss': loss.item()})
                
        val_epoch_loss = val_loss / len(val_loader.dataset)
        accuracy = accuracy_score(all_labels, all_preds)
        precision = precision_score(all_labels, all_preds)
        recall = recall_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds)
        roc_auc = roc_auc_score(all_labels, all_probs)
        
        print(f"Epoch {epoch+1} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_epoch_loss:.4f}")
        print(f"Metrics: Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}, ROC-AUC: {roc_auc:.4f}")

        metrics_data = {
            "epoch": epoch + 1,
            "train_loss": epoch_loss,
            "val_loss": val_epoch_loss,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "roc_auc": roc_auc
        }
        with open(os.path.join(save_dir, 'latest_metrics.json'), 'w') as f:
            json.dump(metrics_data, f, indent=4)

        if val_epoch_loss < best_val_loss:
            best_val_loss = val_epoch_loss
            torch.save(model.state_dict(), os.path.join(save_dir, 'best_model.pth'))
            print(">>> Saved current best model.")
            
    return model

# --- Main Execution ---
DATA_DIR = 'data/Combined_Dataset'
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
FREEZE_BASE_EPOCHS = 5
VAL_SPLIT = 0.15
NUM_WORKERS = 2
CHECKPOINT_DIR = '/content/drive/MyDrive/deepfake_checkpoints'

device = get_device()
print(f"Device: {device}")

full_dataset = DeepfakeDataset(data_root=DATA_DIR, transform=get_train_transforms(), extract_face=True)

if len(full_dataset) == 0:
    raise ValueError("❌ DATASET IS EMPTY! Please check if Step 3 (Download) ran correctly.")

val_size = int(len(full_dataset) * VAL_SPLIT)
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
val_dataset.dataset.transform = get_val_transforms()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train samples: {train_size}")
print(f"Val samples: {val_size}")

model = TransferDeepfakeModel(target_model='efficientnet_b0', freeze_base=True, pretrained=True)

model = train_model(model=model, train_loader=train_loader, val_loader=val_loader, epochs=FREEZE_BASE_EPOCHS, lr=LEARNING_RATE, device=device, save_dir=CHECKPOINT_DIR)

if EPOCHS > FREEZE_BASE_EPOCHS:
    remaining = EPOCHS - FREEZE_BASE_EPOCHS
    model.unfreeze_base_model(num_layers=3)
    model = train_model(model=model, train_loader=train_loader, val_loader=val_loader, epochs=remaining, lr=LEARNING_RATE * 0.1, device=device, save_dir=CHECKPOINT_DIR)

print(f"Training complete! Best model saved to: {os.path.join(CHECKPOINT_DIR, 'best_model.pth')}")